# Step 6: Training 2-Stage Neural Network (MLP & LSTM)

Model 2-stage menggunakan arsitektur **Multi-Layer Perceptron (MLP)** dan **Long Short-Term Memory (LSTM)**:

| Stage | Input | Output | Model |
|-------|-------|--------|-------|
| Stage 1 | 98 fitur | Suit (C/D/H/S/N) | MLP atau LSTM |
| Stage 2 | 98 fitur | Kategori (partscore/game/small_slam/grand_slam) | MLP atau LSTM |

Level final = konversi kategori+suit ke level minimum yang valid.

**Tersedia dua pilihan model:**
- **MLP (Multi-Layer Perceptron)**: Lebih cocok untuk tabular features. Arsitektur: Input → Dense(256) → Dense(128) → Dense(64) → Output
- **LSTM (Long Short-Term Memory)**: Untuk temporal/sequence features. Features direshape menjadi sequence.

**Prasyarat:** Jalankan `05_dataset.ipynb` terlebih dahulu.

In [1]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

DATA_PROCESSED = Path(ROOT) / "data" / "processed"
RESULTS        = Path(ROOT) / "results"
MLP_DIR        = RESULTS / "metrics" / "mlp"
LSTM_DIR       = RESULTS / "metrics" / "lstm"

DATASET_CSV  = DATA_PROCESSED / "bridge_dataset.csv"
SPLIT_PATH   = RESULTS / "metrics" / "dataset_split.pkl"

print(f"Root proyek : {ROOT}")
print("Setup selesai.")

Root proyek : d:\SkripsiBBO
Setup selesai.


## 6.1 Load Dataset Split

In [2]:
from features import FEATURE_COLS
from model import prepare_features
from sklearn.model_selection import train_test_split

if SPLIT_PATH.exists():
    split_data = joblib.load(SPLIT_PATH)
    X_train      = split_data["X_train"]
    X_test       = split_data["X_test"]
    y_suit_train = split_data["y_suit_train"]
    y_suit_test  = split_data["y_suit_test"]
    y_cat_train  = split_data["y_cat_train"]
    y_cat_test   = split_data["y_cat_test"]
    print(f"Split data dimuat dari: {SPLIT_PATH}")
else:
    # Rebuild split dari dataset CSV
    print(f"Split file tidak ditemukan, rebuild dari {DATASET_CSV}")
    df = pd.read_csv(DATASET_CSV)
    df = df.dropna(subset=["best_contract_strain", "best_contract_category"])
    available_feat = [c for c in FEATURE_COLS if c in df.columns]
    X      = prepare_features(df, available_feat)
    y_suit = df["best_contract_strain"]
    y_cat  = df["best_contract_category"]
    X_train, X_test, y_suit_train, y_suit_test = train_test_split(
        X, y_suit, test_size=0.2, stratify=y_suit, random_state=42
    )
    _, _, y_cat_train, y_cat_test = train_test_split(
        X, y_cat, test_size=0.2, stratify=y_suit, random_state=42
    )

print(f"Train : {X_train.shape[0]} sampel")
print(f"Test  : {X_test.shape[0]} sampel")
print(f"Fitur : {X_train.shape[1]}")

Split data dimuat dari: d:\SkripsiBBO\results\metrics\dataset_split.pkl
Train : 4614 sampel
Test  : 1154 sampel
Fitur : 98


## 6.2 Training Model MLP

In [3]:
from model import train, save_model, MLP_MODEL_PATH

# Train MLP
model_mlp = train(X_train, y_suit_train, y_cat_train, model_type="mlp")
save_model(model_mlp, MLP_MODEL_PATH)

Training 2-Stage MLP Model
Building Stage 1 (suit predictor) - MLP...
Training Stage 1 (suit)...
Building Stage 2 (category predictor) - MLP...
Training Stage 2 (category)...
Training selesai.

Model disimpan ke results\metrics\mlp


## 6.3 Training Model LSTM

In [4]:
from model import train, save_model, LSTM_MODEL_PATH

# Train LSTM
model_lstm = train(X_train, y_suit_train, y_cat_train, model_type="lstm")
save_model(model_lstm, LSTM_MODEL_PATH)

Training 2-Stage LSTM Model
Building Stage 1 (suit predictor) - LSTM...
Training Stage 1 (suit)...


KeyboardInterrupt: 

## 6.4 Sanity Check: Prediksi di Training Set

In [ ]:
print("=== MLP Predictions ===")
pred_train_mlp = model_mlp.predict(X_train.head(10))
actual = pd.DataFrame({
    "suit_true":  y_suit_train.head(10).values,
    "cat_true":   y_cat_train.head(10).values,
})
comparison_mlp = pd.concat([actual.reset_index(drop=True), pred_train_mlp.reset_index(drop=True)], axis=1)
comparison_mlp["suit_ok"] = comparison_mlp["suit_true"] == comparison_mlp["pred_suit"]
comparison_mlp["cat_ok"]  = comparison_mlp["cat_true"]  == comparison_mlp["pred_category"]
comparison_mlp

In [ ]:
print("=== LSTM Predictions ===")
pred_train_lstm = model_lstm.predict(X_train.head(10))
comparison_lstm = pd.concat([actual.reset_index(drop=True), pred_train_lstm.reset_index(drop=True)], axis=1)
comparison_lstm["suit_ok"] = comparison_lstm["suit_true"] == comparison_lstm["pred_suit"]
comparison_lstm["cat_ok"]  = comparison_lstm["cat_true"]  == comparison_lstm["pred_category"]
comparison_lstm

## 6.5 Preview Feature Importance

In [ ]:
print("=== MLP Feature Importance ===")
imp_suit_mlp, imp_cat_mlp = model_mlp.feature_importance()

print("Top 10 Fitur Terpenting — Stage 1 (Suit):")
print(imp_suit_mlp.head(10).to_string())
print()
print("Top 10 Fitur Terpenting — Stage 2 (Category):")
print(imp_cat_mlp.head(10).to_string())

In [ ]:
print("=== LSTM Feature Importance ===")
imp_suit_lstm, imp_cat_lstm = model_lstm.feature_importance()

print("Top 10 Fitur Terpenting — Stage 1 (Suit):")
print(imp_suit_lstm.head(10).to_string())
print()
print("Top 10 Fitur Terpenting — Stage 2 (Category):")
print(imp_cat_lstm.head(10).to_string())

---
## Output

File yang dihasilkan:
- `results/metrics/mlp/`: Model MLP tersimpan
- `results/metrics/lstm/`: Model LSTM tersimpan

**Langkah berikutnya:** Buka `07_evaluasi.ipynb` untuk evaluasi kedua model.